---
title: "Tokenizer：BPE、WordPiece与Unigram分词算法"
author: "逃之夭夭"
date: "2026-04-13"
categories: [tokenizer, deep learning, llm]
image: "images/index.jpg"
---

> *"我的语言的边界，就是我的世界的边界。"*          ———— 路德维希·维特根斯坦

语言模型在理解世界之前，首先要学会切分语言——而这件事远比想象中复杂。

"hugging" 该拆成 `hug` + `##ging` 还是 `hu` + `##gging`？
中文的"自然语言处理"又该怎么断？

这些问题的答案，取决于分词器（Tokenizer）的算法选择。
现代 NLP 模型普遍采用子词（Subword）分词策略，在字符级和词表级之间寻找平衡。

目前主流的三种子词算法是：

| 算法 | 代表模型 |
|------|----------|
| BPE（Byte-Pair Encoding） | GPT-2、LLaMA、RoBERTa |
| WordPiece | BERT、DistilBERT |
| Unigram | T5、XLNet、AlBERT |

它们的训练哲学截然相反：BPE 和 WordPiece **从小到大**合并，Unigram **从大到小**裁剪。


在了解三种算法之前，先从高层视角概览分词流程的步骤。

## 归一化和预分词

<div class="flex justify-center">
<img class="block dark:hidden" src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter6/tokenization_pipeline.svg" alt="The tokenization pipeline." style="max-width: 600px; width: 100%; height: auto;">
</div>

> 各个步骤因分词算法各异

- Normalization：对文本执行的任何清理操作，包括去除重音字符、空格或将文本转化为小写等规则
- Pre-tokenization：对文本预分词，包括基于空格或者标点符号等规则
- Postprocessor：添加特殊token字符等

## BPE（Byte-Pair Encoding）

一句话：**反复找最频繁的相邻对，合并它，记下来。**

从字符出发，每一轮找语料中出现最频繁的相邻符号对并合并，
合并规则按顺序记录——推理时，新词也按同样的顺序"回放"这些规则。



### Byte-level BPE
普通BPE以字符为最小单元，遇到词表外的字符只能输出[UNK]。 GPT-2改用字节作为最小单元——256个字节覆盖一切Unicode，任何文本都能无损编码，彻底消灭了[UNK]。

<div style="font-family: monospace; background:#f6f8fa; border-radius:8px; padding:1.2em 1.5em; line-height:2;">
  <div><span style="color:#888">普通 BPE：</span>
    "é" 不在词表 → <span style="color:#e05252">[UNK]</span></div>
  <div><span style="color:#888">Byte-level：</span>
    "é" → 字节 <code>0xC3 0xA9</code> → 合并为 token <code>Ġé</code> ✅</div>
</div>

## WordPiece

一句话：**合并"在一起比分开更意外"的符号对。**

与 BPE 的区别在于合并标准——不选最高频的对，
而是选**互信息最高**的对：两个符号单独出现很少，
但总是结伴出现，才值得合并。

$$\text{score}(A, B) = \frac{\text{freq}(AB)}{\text{freq}(A) \times \text{freq}(B)}$$

<div style="font-family: monospace; background:#f6f8fa; border-radius:8px; padding:1.2em 1.5em; line-height:2;">
  <div style="color:#888; margin-bottom:0.3em">同样的语料，两种候选合并对：</div>
  <div>
    <code>(e, d)</code> → freq=100，但 e 和 d 各自也很常见
    → score <b style="color:#e05252">低</b>
  </div>
  <div>
    <code>(##ing, ly)</code> → freq=40，但各自单独几乎不出现
    → score <b style="color:#2a9d5c">高</b> ✅
  </div>
</div>


推理时不用合并规则，只用词表：从词头找**最长匹配子词**，
切不开的整词直接标为 `[UNK]`。

## Unigram

一句话：**从大词表开始，删去"删了也没太大损失"的 token。**

每个 token 被赋予一个概率（由频率估计），
一种分词方案的"好坏"用概率之积衡量：

$$P(\text{方案}) = \prod_{i} P(t_i)$$

每轮评估：如果删掉某个 token，语料的总 loss 增加多少？
增加越少，说明它越可有可无，优先删除。

```{mermaid}
graph LR
  A["大词表\n(~1000 tokens)"] -->|"删除贡献最小的 20%"| B["缩小词表"]
  B -->|"再删 20%"| C["..."]
  C -->|"达到目标大小"| D["最终词表\n(保留基础字符)"]
```

推理时用 **Viterbi 算法**在所有可能分词路径中找概率最大的一条。

## 结束语

BERT 上线时上下文窗口只有 512 token，如今 LLM 动辄支持 200K+，`vocab_size` 也从 3 万膨胀至 10 万+。

词表越大，同样的文本切出的 token 越少，模型能"看"得越远——分词器的设计，从一开始就在影响模型的上限。

从 BPE 的贪心合并，到 WordPiece 的互信息驱动，再到 Unigram 的概率裁剪，三种算法殊途同归：**用最少的 token，表达最完整的语义。**


## References

1. [Hugging Face NLP Course - Chapter 6](https://huggingface.co/learn/llm-course/en/chapter6/1)